In [ ]:
import os
import logging
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

# -------------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Replace with your actual CSV file path
INPUT_CSV = "/home/henrykabuye/Ransap/Reading_IJCNN/temp_dataset.csv"

# Expected column names (edit as needed to match your dataset)
# Example here: we're expecting columns like "GPA", "LBA", "size", etc.
COLUMN_NAMES = [
    "GPA", "LBA", "size", "entropy", "type", "timestamp",
    "GPA_variance", "LBA_variance", "category", "label", "file_source"
]

# Where to save smaller processed files
OUTPUT_DIR = "processed_chunks"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Batch and chunk sizes
BATCH_SIZE = 32
CHUNK_SIZE = 100_000  # Adjust based on your RAM constraints

# Choose a Transformer model. "bert-base-uncased" is a good default.
MODEL_NAME = "bert-base-uncased"

# -------------------------------------------------------------------------
# Model / Tokenizer Setup
# -------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# -------------------------------------------------------------------------
# Utility Functions
# -------------------------------------------------------------------------
def create_text_representation(row: pd.Series) -> str:
    """
    Converts a row from the dataset into a structured text description
    so BERT can process it semantically.
    """
    # Build a plain-English string from numeric columns
    # This is just an example—tailor it to your domain!
    feature_texts = []
    for col, value in row.items():
        if pd.isna(value):  # skip missing values
            continue
        
        col_lower = col.lower()
        if col_lower in ["gpa", "lba"]:
            feature_texts.append(f"Access at {col.upper()} {value}")
        elif col_lower == "size":
            feature_texts.append(f"Size: {value} bytes")
        elif col_lower == "entropy":
            feature_texts.append(f"Entropy level: {value}")
        elif col_lower == "type":
            feature_texts.append(f"Type: {value}")
        elif col_lower in ["category", "label"]:
            feature_texts.append(f"Category: {value}")
        else:
            # For timestamps, file_source, or anything else
            feature_texts.append(f"{col}: {value}")
    
    # Join into a sentence or short paragraph
    return ". ".join(feature_texts)

def extract_embeddings_batch(texts: list, batch_size: int = BATCH_SIZE) -> np.ndarray:
    """
    Given a list of text entries, tokenize them in batches and extract embeddings.
    Returns an N x D numpy array where N = len(texts) and D = model embedding dimension.
    """
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting embeddings"):
        batch_texts = texts[i : i + batch_size]

        # Tokenize in batch
        inputs = tokenizer(batch_texts, return_tensors="pt",
                           truncation=True, padding=True, max_length=256)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Extract embeddings
        with torch.no_grad():
            outputs = model(**inputs)
        
        # outputs.last_hidden_state: [batch_size, seq_len, hidden_dim]
        # Here we do mean pooling over the sequence length
        batch_embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        embeddings.append(batch_embeddings)
    
    # Concatenate all batches
    return np.vstack(embeddings)

def process_large_dataset(input_csv: str):
    """
    Processes a large CSV in chunks, converts each row to text, extracts embeddings,
    and writes out a smaller CSV containing the new embeddings.
    """
    logging.info(f"Processing dataset in chunks from: {input_csv}")

    chunk_idx = 0
    # The user might have no headers or partial columns, so adjust read_csv params:
    for chunk in tqdm(
        pd.read_csv(input_csv, header=None, names=COLUMN_NAMES, chunksize=CHUNK_SIZE),
        desc="Reading CSV by chunks"
    ):
        logging.info(f"Processing chunk {chunk_idx} with {len(chunk)} rows")

        # Create a text representation from each row
        chunk["text_representation"] = chunk.apply(create_text_representation, axis=1)

        # Extract embeddings
        texts = chunk["text_representation"].tolist()
        embeddings = extract_embeddings_batch(texts, BATCH_SIZE)

        # Save embeddings as a new column (list-of-lists or array)
        chunk["llm_embeddings"] = list(embeddings)

        # Save each chunk to CSV
        out_file = os.path.join(OUTPUT_DIR, f"processed_chunk_{chunk_idx}.csv")
        chunk.to_csv(out_file, index=False)
        logging.info(f"Saved chunk {chunk_idx} to {out_file}")

        chunk_idx += 1

    logging.info("All chunks processed.")

# -------------------------------------------------------------------------
# Main
# -------------------------------------------------------------------------
if __name__ == "__main__":
    process_large_dataset(INPUT_CSV)

    


2025-03-24 15:23:11.160818: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742829791.224085  235112 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742829791.243506  235112 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-24 15:23:11.375031: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-24 15:23:14,211 - INFO - Processing dataset in chunks from: /home/henrykabuye/Ransap/Reading_IJCNN/temp_dataset.csv
R

In [1]:
import os

def remove_duplicate_headers(infile, outfile):
    """
    Read 'infile' line by line, write to 'outfile' but skip any lines that
    match the *first* header row (except the actual first line).
    """
    with open(infile, 'r', encoding='utf-8', errors='replace') as fin, \
         open(outfile, 'w', encoding='utf-8') as fout:
        
        first_line = None
        line_count = 0
        
        for line in fin:
            if line_count == 0:
                # The very first line is the real header, write it
                first_line = line
                fout.write(line)
            else:
                # If line exactly matches the header, skip it
                if line == first_line:
                    continue
                else:
                    fout.write(line)
            line_count += 1


# Example usage:
infiles = [
   "/home/henrykabuye/Ransap/Reading_IJCNN/temp_dataset.csv",
   "/home/henrykabuye/Ransap/Reading_IJCNN/temp_dataset.csv",
   # etc.
]
for path in infiles:
    dirname, filename = os.path.split(path)
    base, ext = os.path.splitext(filename)
    out_path = os.path.join(dirname, f"{base}_clean{ext}")
    
    remove_duplicate_headers(path, out_path)
    print(f"Created cleaned file: {out_path}")


Created cleaned file: /home/henrykabuye/Ransap/Reading_IJCNN/temp_dataset_clean.csv
Created cleaned file: /home/henrykabuye/Ransap/Reading_IJCNN/temp_dataset_clean.csv
